In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_Burari_Crossing_Delhi_IMD_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,287.0,151.0,NaN,118.0,227.0,298.0,NaN,61.0,NaN,160.0,355.0,NaN
1,2,283.0,148.0,98.0,151.0,202.0,198.0,87.0,66.0,NaN,197.0,358.0,NaN
2,3,NaN,204.0,NaN,NaN,268.0,156.0,91.0,63.0,NaN,148.0,394.0,NaN
3,4,379.0,NaN,137.0,179.0,274.0,218.0,56.0,52.0,NaN,214.0,NaN,NaN
4,5,313.0,NaN,146.0,184.0,337.0,257.0,76.0,46.0,59.0,154.0,372.0,170.0
5,6,NaN,138.0,NaN,175.0,242.0,150.0,56.0,41.0,88.0,123.0,363.0,NaN
6,7,334.0,223.0,194.0,NaN,338.0,211.0,54.0,39.0,65.0,113.0,382.0,259.0
7,8,334.0,140.0,157.0,185.0,214.0,229.0,56.0,NaN,80.0,191.0,NaN,343.0
8,9,329.0,136.0,NaN,NaN,147.0,167.0,70.0,78.0,98.0,204.0,340.0,168.0
9,10,273.0,285.0,163.0,NaN,146.0,155.0,135.0,57.0,79.0,NaN,340.0,220.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,287.00000,151.00000,143.451613,156.827586,227.000000,298.000000,78.5,61.000000,83.125,160.000000,355.000000,257.133333
1,2,283.00000,148.00000,143.451613,151.000000,202.000000,198.000000,87.0,66.000000,83.125,197.000000,358.000000,257.133333
2,3,290.40625,204.00000,143.451613,156.827586,268.000000,156.000000,91.0,63.000000,83.125,148.000000,394.000000,257.133333
3,4,379.00000,162.62963,137.000000,179.000000,274.000000,218.000000,56.0,52.000000,83.125,214.000000,307.541667,257.133333
4,5,313.00000,162.62963,146.000000,184.000000,337.000000,257.000000,76.0,46.000000,59.000,154.000000,372.000000,170.000000
5,6,290.40625,138.00000,143.451613,175.000000,242.000000,150.000000,56.0,41.000000,88.000,123.000000,363.000000,257.133333
6,7,334.00000,223.00000,194.000000,156.827586,338.000000,211.000000,54.0,39.000000,65.000,113.000000,382.000000,259.000000
7,8,334.00000,140.00000,157.000000,185.000000,214.000000,229.000000,56.0,64.172414,80.000,191.000000,307.541667,343.000000
8,9,329.00000,136.00000,143.451613,156.827586,147.000000,167.000000,70.0,78.000000,98.000,204.000000,340.000000,168.000000
9,10,273.00000,162.62963,163.000000,156.827586,146.000000,155.000000,78.5,57.000000,79.000,210.030303,340.000000,220.000000
